# Drug Therapeutic use with geometric deep learning and human centered design

This project focuses on AI-driven drug repurposing using Graph Neural Networks (GNNs). The system is designed to model complex biological entities as graphs and predict new drug-disease associations by analyzing network-based relationships, which can be used in a wide range of applications such as precision medicine and identifying new uses for existing medications
The main tasks of the project are:   
1-Preprocessing biological datasets and knowledge graphs like PrimeKG.  
2-Representing drugs and diseases as nodes within a biological graph.  
3-Implementing and training GNN architecture (such as TxGNN) for link prediction.  
4-Evaluating model performance using metrics like AUROC and AUPRC.  
5-Predicting high-probability drug candidates for specific diseases.  


## Step 1: Understand the data (Hetionet structure)

Hetionet contains 11 types of Meta nodes and 24 types of edges.  
 For our project,we will focus on a core sub-network that connects compounds to diseases, passing through proteins (genes)  
   The Essential Node Types (kind or node_type)  
       
       Compound ⚠️ (This represents your Drugs, using DrugBank IDs like DB00338)  

       Disease ⚠️ (This represents your Diseases, using Disease Ontology IDs like DOID:8398)  

       Gene ⚠️ (This represents your Proteins, using Entrez Gene IDs like Gene::5)    

       Anatomy: This represents human Organs and Tissues, using UBERON ontology IDs like UBERON:0002048 (e.g., Lung or Liver).

      Pathway: This represents Biological Pathways, using Pathway Commons IDs like PC7_1124 (e.g., Cellular signaling cascades).

      Side Effect: This represents Adverse Drug Reactions, using UMLS CUI IDs like C0027497 (e.g., Nausea or Bradycardia).

      Symptom: This represents Clinical Symptoms of diseases, using MeSH IDs like Mesh:D003967 (e.g., Diarrhea or Pain).

      Pharmacologic Class: This represents the Mechanism of Action Classes of drugs, using NDF-RT IDs like N0000175574 (e.g., Proton Pump Inhibitors).

      Biological Process: This represents broad Cellular Objectives, using Gene Ontology IDs like GO:0006915 (e.g., Apoptotic cell death program).

      Molecular Function: This represents specific Biochemical Tasks, using Gene Ontology IDs like GO:0004672 (e.g., Protein kinase catalytic activity).

      Cellular Component: This represents the Subcellular Locations where proteins work, using Gene Ontology IDs like GO:0005737 (e.g., Cytoplasm or Nucleus) 

   The Essential Edge Types (relation)  

         CtD (Compound–treats–Disease): ⚠️ This is My goldmine. (My Target)

         CbG (Compound–binds–Gene): ⚠️ Shows which protein target a drug physically binds to.

         DaG (Disease–associates–Gene): ⚠️Shows which genes are known to be involved in disease.  

         GPPw (Gene Participate Pathway) : ⚠️show gena participate in which pathway

         GcG (Gene–interacts–Gene): ⚠️ Protein-Protein Interactions (PPI).

         CpD: Compound palliates Disease

         CseSE: Compound causes Side Effect

         DbA: Disease localizes to Anatomy

         DlD: Disease resembles Disease

         DlA Disease–localizes–Anatomy  

         DrD Disease–resembles–Disease

         CrC Compound–resembles–Compound

         DsD: Disease associates with Disease

         CuG: Compound upregulates Gene

         CdG: Compound downregulates Gene

         DuG: Disease upregulates Gene

         DdG: Disease downregulates Gene

         GgG: Gene interacts with Gene

         GpBP: Gene participates in Biological Process

         GpMF: Gene participates in Molecular Function

         GpCC: Gene participates in Cellular Component

## Step 1: Read the data

In [1]:
# Parquet Format is easier and faster and save storage rather than CSV or TSV
import os
import pandas as pd
nodes_tsv = os.path.join("hetionet-data", "hetnet", "tsv", "hetionet-v1.0-nodes.tsv")
edges_gz = os.path.join("hetionet-data", "hetnet", "tsv", "hetionet-v1.0-edges.sif.gz")
nodes_parquet = os.path.join("hetionet-data", "hetnet", "tsv", "nodes.parquet")
edges_parquet = os.path.join("hetionet-data", "hetnet", "tsv", "edges.parquet")

try:
    print("Reading raw data files")
    nodes_df = pd.read_csv(nodes_tsv, sep="\t", low_memory=False)
    edges_df = pd.read_csv(edges_gz, sep="\t", compression="gzip", low_memory=False)
    
    print("Saving to Parquet format for ultimate speed")
    nodes_df.to_parquet(nodes_parquet, engine="fastparquet")
    edges_df.to_parquet(edges_parquet, engine="fastparquet")
    print("All files converted to Parquet successfully!")
    print("=" * 60)

    print("Unique Node kinds:")
    print(nodes_df['kind'].unique())
    print("\nUnique Edge metaedges:")
    print(edges_df['metaedge'].unique())
    print("-" * 60)
    
    print(f"🔹 First five nodes:\n{nodes_df.head()}\n")
    print(f"🔹 Pathway Nodes Example:\n{nodes_df[nodes_df['kind'] == 'Pathway'].head()}\n")
    print("-" * 60)
    print(f"🔹 First five edges:\n{edges_df.head()}\n")
    print(f"🔹 CrC Edges Example:\n{edges_df[edges_df['metaedge'] == 'CrC'].head()}\n")

except FileNotFoundError as e:
    print(f" Error: Please check the file path. System cannot find the file: {e}")
except Exception as e:
    print(f" Another error happened: {e}")

Reading raw data files
Saving to Parquet format for ultimate speed
All files converted to Parquet successfully!
Unique Node kinds:
<StringArray>
[            'Anatomy',  'Biological Process',  'Cellular Component',
            'Compound',             'Disease',                'Gene',
  'Molecular Function',             'Pathway', 'Pharmacologic Class',
         'Side Effect',             'Symptom']
Length: 11, dtype: str

Unique Edge metaedges:
<StringArray>
['GpBP',  'GiG',  'CrC',  'DdG',  'DpS',  'DlA',  'CtD',  'CbG',  'CuG',
  'DrD',  'DaG',  'CpD',  'AdG',  'AuG',  'GcG', 'GpMF', 'PCiC', 'GpCC',
 'Gr>G',  'CdG',  'DuG', 'GpPW', 'CcSE',  'AeG']
Length: 24, dtype: str
------------------------------------------------------------
🔹 First five nodes:
                        id                       name     kind
0  Anatomy::UBERON:0000002             uterine cervix  Anatomy
1  Anatomy::UBERON:0000004                       nose  Anatomy
2  Anatomy::UBERON:0000006        islet of Langer

**Hetionet Data Processing Script**

This script is designed to efficiently load, convert, and explore the Hetionet dataset through a structured workflow consisting of three main stages: data loading, format optimization, and exploratory data analysis (EDA).

1.✅**Data Loading**


The script imports essential libraries such as os and pandas.
It defines file paths for:
Nodes dataset (TSV format): contains biological entities like genes, diseases, and pathways.
Edges dataset (compressed .gz format): represents relationships between entities.
Both datasets are loaded into pandas DataFrames using:
read_csv() with tab separation (sep="\t") for TSV files.
Automatic decompression using compression="gzip" for edge data.


**✅ 2. Data Format Conversion (Optimization)**


To improve performance and storage efficiency, the datasets are converted into Parquet format, a columnar storage format optimized for speed.
Conversion is done using:
nodes_df.to_parquet() for nodes data.
edges_df.to_parquet() for edges data.
The fastparquet engine is used for writing.
This step significantly enhances future read/write operations compared to CSV or TSV formats.


** 3. **✅Exploratory Data Analysis (EDA)**

The script performs basic data exploration to understand structure and relationships:

Identifies unique values:
Node types using nodes_df['kind'].unique()
Relationship types using edges_df['metaedge'].unique()
Displays sample data:
First 5 rows of nodes and edges using .head()
Filtered examples such as:
Nodes where kind == 'Pathway'
Edges where metaedge == 'CrC'

These steps help reveal the dataset structure and distribution.


 4. ✅**Error Handling**

The script includes robust error handling mechanisms:

FileNotFoundError: triggered when input files are missing or paths are incorrect.
Generic Exception: captures any unexpected runtime errors.

example for node: have Id and human name like "uterine cervix" and kind "anatomy 
sec node : have id of disease symptom called "Symptom::D064250" and human name "Hypertriglyceridemic Waist" and kind called symptom

example for edge 
source "like id  // metaedge GPBP gene participating in biological process  then 
target Go to biological process

$$\text{Compound} \xrightarrow{\text{CbG}} \text{Gene} \xrightarrow{\text{GrPW}} \text{Pathway} \xrightarrow{\text{GrPW}^{-1}} \text{Gene} \xrightarrow{\text{DaG}} \text{Disease}$$

%pip install tqdm
!pip install torch_geometric
!pip install torch --index-url https://download.pytorch.org/whl/cu121 --default-timeout=1000

## Dreamwalk Pipeline

In [2]:
import torch


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 الـ كود شغال حالياً على: {device} ({torch.cuda.get_device_name(0)})")

🚀 الـ كود شغال حالياً على: cuda (NVIDIA RTX 2000 Ada Generation)



**📄 Code Explanation**

This snippet is used to configure the computing device for PyTorch operations and ensure that the code runs on the most efficient available hardware (GPU or CPU).

✅. **Library Import**

The script starts by importing the PyTorch library:

torch is a deep learning framework used for tensor computations and model training.
✅. **Device Selection (CPU vs GPU)**

The code then determines the best available device for computation:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.cuda.is_available() checks whether a GPU (CUDA-enabled) is available.
If a GPU is available → the device is set to "cuda".
Otherwise → the computation falls back to "cpu".
This ensures that the code can run efficiently on different environments without manual changes.

✅. **Printing Device Information**

Finally, the script prints the selected device:

print(f" {device} ({torch.cuda.get_device_name(0)})")
device shows whether the code is running on CPU or GPU.
torch.cuda.get_device_name(0) retrieves the name of the GPU.

⚠️ Note: This assumes a GPU is available. If the code runs on CPU only, calling get_device_name(0) may raise an error unless handled separately.

In [3]:
"""
DREAMwalk — Optimized, Leak-Free, GPU-Accelerated Pipeline
============================================================
Fixes vs. the original script:

1. ZERO DATA LEAKAGE
   - Jaccard similarity matrices (drug_sim / disease_sim) are now computed
     INSIDE each CV fold, from G_train only (test CtD edges removed first).
   - No cached global similarity matrices are reused across folds.

2. SPEED
   - Graph adjacency is represented as a SciPy/CuPy sparse CSR matrix
     instead of a NetworkX graph -> Jaccard similarity for the (small)
     Compound and Disease subsets is computed as ONE dense matrix
     multiplication (GPU via CuPy if available, else NumPy/SciPy).
   - Random walks are fully vectorized: every step advances ALL walks at
     once using padded neighbor arrays + NumPy batched sampling, instead
     of a per-walk, per-node Python loop. (Implements the exact uniform
     2nd-order-neutral case p=q=1 from CONFIG; see `biased_step` for the
     general p!=1/q!=1 fallback.)
   - predict_all_pairs builds the full Compound x Disease feature matrix
     in one vectorized NumPy block and scores it in big XGBoost batches
     instead of a python loop per drug.

CONFIG keys are unchanged so this is a drop-in replacement.
"""

import os
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy import sparse as sp #SparseGraph replace nx.Graph

from gensim.models import Word2Vec
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score
from sklearn.preprocessing import StandardScaler
import xgboost as xgb

import warnings
warnings.filterwarnings("ignore")

# Optional GPU backend for the similarity matrix multiply.
try:
    import cupy as cp
    import cupyx.scipy.sparse as csp
    GPU_AVAILABLE = True
except Exception:
    GPU_AVAILABLE = False

 **1.✅ Zero Data Leakage (Core Improvement)**
The pipeline ensures strict cross-validation safety.
All similarity computations (drug–drug and disease–disease Jaccard similarity) are performed inside each training fold only.
Test edges (CtD interactions) are fully removed before any computation.
This guarantees that no test information leaks into training, preserving model validity and unbiased evaluation.

**✅ 2. Performance Optimizations**

The script introduces multiple efficiency improvements:

Graph Representation Upgrade
Uses SciPy / CuPy CSR sparse matrices instead of NetworkX.
Reduces memory usage and speeds up matrix operations.
Fast Similarity Computation
Jaccard similarity is computed using dense matrix multiplication.
Supports:
CPU acceleration via NumPy/SciPy
GPU acceleration via CuPy (if available)
Vectorized Random Walks
Eliminates Python loops completely.
Walks are processed in parallel using:
padded neighbor arrays
batched sampling
Implements uniform second-order neutral random walk (p = q = 1).
Optimized Prediction Pipeline
Full Compound × Disease feature matrix is built in a single vectorized step.
XGBoost inference is executed in large batches instead of per-edge prediction, improving speed significantly.


**✅ 3. Configuration Compatibility**
The original CONFIG remains unchanged.
The script is a drop-in replacement, meaning:
No API changes
No parameter modifications required
Ensures easy integration into existing workflows.

**✅ 4. Libraries and Dependencies**

The pipeline is built using standard scientific and ML libraries:

NumPy / Pandas → data handling
SciPy (sparse) → graph representation
Gensim (Word2Vec) → embedding learning from random walks
Scikit-learn → evaluation and cross-validation
XGBoost → final predictive model
CuPy (optional) → GPU acceleration for sparse matrix operations

**✅ 5. GPU Acceleration Support**

The script includes automatic hardware detection:
If CuPy is available + compatible GPU exists:
Computation runs on GPU
Otherwise:
System automatically falls back to CPU
This ensures:
High performance on GPUs
Full portability across environments


**Final Outcome**

The optimized DREAMwalk pipeline delivers:

✔ No data leakage

✔ Faster graph processing

✔ Scalable vectorized computations

✔ Optional GPU acceleration

✔ Fully compatible drop-in replacement design


In [ ]:
# 1. CONFIGURATION (unchanged architecture)

CONFIG = {
    "walk_length":      100, # 
    "num_walks":        10, #
    "teleport_prob":    0.5,
    "p":                1.0,
    "q":                1.0,
    "embed_dim":        128,
    "window":           5,
    "min_count":        1,
    "sg":               1,
    "workers":          4,
    "epochs":           5,
    "n_folds":          10, 
    "xgb_params": {
        "n_estimators":     300,
        "max_depth":        6,
        "learning_rate":    0.05,
        "subsample":        0.8,
        "colsample_bytree": 0.8,
        "eval_metric":      "logloss",
        "random_state":     42,
        "n_jobs":           -1,
        "tree_method":      "hist",
        "device":           "cuda", # used for GPU
    },
    "random_seed": 42,
    "cache_dir": "./dreamwalk_cache",
}

random.seed(CONFIG["random_seed"])
np.random.seed(CONFIG["random_seed"])
os.makedirs(CONFIG["cache_dir"], exist_ok=True)

📄 Code Explanation — Configuration Setup

This section defines the global configuration parameters for the DREAMwalk pipeline, ensuring reproducibility, consistency, and centralized control over all model and training hyperparameters.

 **1.Random Walk Parameters**

These control how graph random walks explore the network:

walk_length = 100 → number of steps per walk
num_walks = 10 → number of walks per node
teleport_prob = 0.5 → probability of random teleportation (enhances exploration)
p = 1.0, q = 1.0 → neutral Node2Vec-style parameters (unbiased random walk behavior)

 **2. Embedding Learning (Word2Vec)**

These parameters define how node embeddings are learned from random walks:

embed_dim = 128 → embedding vector size
window = 5 → context window size
min_count = 1 → includes all nodes regardless of frequency
sg = 1 → Skip-Gram model
workers = 4 → parallel CPU threads
epochs = 5 → training iterations

 **3. Cross-Validation Setup**

n_folds = 10 → uses Stratified K-Fold cross-validation for evaluation

 **4. XGBoost Model Parameters**

These control the final supervised learning stage:

n_estimators = 300 → number of boosting rounds
max_depth = 6 → tree complexity control
learning_rate = 0.05 → step size shrinkage
subsample = 0.8 → row sampling per tree
colsample_bytree = 0.8 → feature sampling per tree
eval_metric = "logloss" → optimization objective
random_state = 42 → ensures reproducibility
n_jobs = -1 → uses all CPU cores
tree_method = "hist" → optimized histogram-based training
device = "cuda" → enables GPU acceleration if available

  **5. System-Level Settings**

random_seed = 42 → ensures full reproducibility across runs
cache_dir = "./dreamwalk_cache" → stores intermediate results (e.g., embeddings, graphs, matrices)

 **6. Reproducibility Control**

To ensure consistent outputs across environments:

random.seed(CONFIG["random_seed"])
np.random.seed(CONFIG["random_seed"])
Controls randomness in both Python and NumPy operations
Ensures results are stable and repeatable

 **7. Cache Initialization**

os.makedirs(CONFIG["cache_dir"], exist_ok=True)
Creates a directory for storing intermediate computation results
Prevents errors if the folder already exists
Supports faster re-runs by reusing cached data

**🧠 Final Insight**

This configuration design makes the pipeline:

✔ Highly modular

✔ Easy to tune and experiment with

✔ Fully reproducible

✔ Efficient through caching and parallelism


In [12]:
# 2. DATA LOADING
# ─────────────────────────────────────────────
def load_hetionet_local():
    nodes_df = pd.read_parquet(r"D:\Project Data\hetionet-data\hetnet\tsv\nodes.parquet")
    edges_df = pd.read_parquet(r"D:\Project Data\hetionet-data\hetnet\tsv\edges.parquet")
    treats_df = edges_df[edges_df['metaedge'] == 'CtD'].copy()
    print(f"Nodes: {len(nodes_df):,}")
    print(f"Edges: {len(edges_df):,}")
    print(f"CtD Edges: {len(treats_df):,}")
    return nodes_df, edges_df, treats_df

**1. Function Definition**

 **def load_hetionet_local():t**

Defines a reusable and modular function.
Encapsulates all dataset loading logic in one place.
Improves code organization, readability, and maintainability.

 **2. Loading Preprocessed Data**

The function loads two main datasets stored in Parquet format:

**nodes_df** = pd.read_parquet("...nodes.parquet")
**edges_df** = pd.read_parquet("...edges.parquet")

 **Components:**

**nodes_df** → biological entities (genes, diseases, compounds, pathways)
**edges_df** → relationships between entities in the heterogeneous graph
 Why Parquet?

Faster I/O performance than CSV/TSV
Reduced storage size
Optimized for analytical workloads

**3. Filtering Biological Interactions (CtD)**

**treats_df = edges_df[edges_df['metaedge'] == 'CtD'].copy()**
Extracts only Compound → treats → Disease (CtD) relationships.
This subset represents the main prediction target in drug–disease association tasks.
.copy() ensures the filtered DataFrame is independent and avoids pandas warnings.

**4. Dataset Statistics**

The function prints quick summary statistics:

 **print(f"Nodes: {len(nodes_df):,}")
print(f"Edges: {len(edges_df):,}")
print(f"CtD Edges: {len(treats_df):,}")**

**📌 Reported Information:**

Total number of biological entities (nodes)
Total number of relationships (edges)
Number of CtD interactions (key predictive subset)

✔ Helps verify successful loading
✔ Gives immediate insight into dataset scale

**5. Function Output**
return nodes_df, edges_df, treats_df

The function returns three structured DataFrames:

**nodes_df** → entity-level biological information
**edges_df** → full heterogeneous graph
**treats_df** → filtered drug–disease interaction dataset

**🧠 Final Insight**

This function provides a clean and efficient entry point to the Hetionet dataset by:

✔ Using optimized Parquet loading

✔ Isolating the key prediction task (CtD)

✔ Providing quick dataset diagnostics

✔ Returning structured outputs ready for ML pipelines

In [13]:
# 3. SPARSE GRAPH REPRESENTATION (replaces networkx.Graph)
# ─────────────────────────────────────────────
class SparseGraph:
    """
    Lightweight undirected graph backed by a SciPy CSR boolean adjacency
    matrix, plus padded neighbor arrays for vectorized random walking.
    Replaces nx.Graph for everything this pipeline needs.
    """
    def __init__(self, node_ids):
        self.node_ids = list(node_ids)
        self.id2idx = {n: i for i, n in enumerate(self.node_ids)}
        self.n = len(self.node_ids)
        self.adj = None          # CSR boolean, built via build_from_edges
        self.neighbor_idx = None  # (n, max_deg) int32, -1 padded
        self.neighbor_cnt = None  # (n,) int32

    def build_from_edges(self, src_idx, dst_idx):
        rows = np.concatenate([src_idx, dst_idx])
        cols = np.concatenate([dst_idx, src_idx])
        data = np.ones(len(rows), dtype=np.bool_)
        self.adj = sp.csr_matrix((data, (rows, cols)), shape=(self.n, self.n))
        self.adj.data[:] = True
        self.adj.sort_indices()
        self._build_padded_neighbors()

    def _build_padded_neighbors(self):
        indptr = self.adj.indptr
        indices = self.adj.indices
        deg = np.diff(indptr)
        max_deg = int(deg.max()) if len(deg) else 0
        max_deg = max(max_deg, 1)
        neighbor_idx = np.full((self.n, max_deg), -1, dtype=np.int32)
        for i in range(self.n):
            s, e = indptr[i], indptr[i + 1]
            neighbor_idx[i, : e - s] = indices[s:e]
        self.neighbor_idx = neighbor_idx
        self.neighbor_cnt = deg.astype(np.int32)

    def remove_edges(self, pairs_idx):
        """Return a NEW SparseGraph with the given (i, j) index pairs removed."""
        coo = self.adj.tocoo()
        remove_set = set()
        for i, j in pairs_idx:
            remove_set.add((i, j))
            remove_set.add((j, i))
        mask = np.array(
            [(r, c) not in remove_set for r, c in zip(coo.row, coo.col)]
        )
        new_adj = sp.csr_matrix(
            (coo.data[mask], (coo.row[mask], coo.col[mask])), shape=(self.n, self.n)
        )
        g = SparseGraph(self.node_ids)
        g.adj = new_adj
        g._build_padded_neighbors()
        return g


def build_graph(edges_df, nodes_df):
    node_ids = nodes_df['id'].tolist()
    G = SparseGraph(node_ids)
    src_idx = edges_df['source'].map(G.id2idx).to_numpy()
    dst_idx = edges_df['target'].map(G.id2idx).to_numpy()
    valid = (~pd.isna(src_idx)) & (~pd.isna(dst_idx))
    G.build_from_edges(src_idx[valid].astype(np.int64), dst_idx[valid].astype(np.int64))
    return G


def get_node_kinds(nodes_df):
    return dict(zip(nodes_df['id'], nodes_df['kind']))

**📄Sparse Graph Representation**

This section introduces SparseGraph, a custom high-performance graph structure designed to replace networkx.Graph for large-scale heterogeneous biological networks. It is optimized for memory efficiency, fast traversal, and vectorized random walk operations.

 **1. SparseGraph Class Overview**
class SparseGraph:
Implements a lightweight undirected graph.
Replaces NetworkX to improve scalability.
Uses SciPy CSR sparse matrices as the core adjacency representation.

**Key Advantage:**

Much lower memory usage
Faster matrix-based computations
Better suited for ML pipelines on large biological graphs

 **2. Initialization**
def __init__(self, node_ids):

**Core Components:**

node_ids → list of all graph nodes
id2idx → maps node IDs → integer indices (for matrix operations)
n → total number of nodes

 **Internal Structures:**
adj → CSR sparse adjacency matrix
neighbor_idx → padded neighbor list per node
neighbor_cnt → degree (number of neighbors per node)

 **3. Graph Construction from Edges**
def build_from_edges(self, src_idx, dst_idx):

 **Process:**
Converts edge list into adjacency structure
Adds edges in both directions:
(src → dst)
(dst → src)
Ensures the graph is undirected

**🧠 Optimization:**
Stores adjacency in CSR sparse matrix format
Uses boolean values only (True = connection)
No weights → faster computation
Sorted indices improve traversal efficiency

 **4. Padded Neighbor Representation**
def _build_padded_neighbors(self):

**Purpose:**

Precomputes neighbor lists into a fixed-size structure for vectorized operations.

 **How it works:**
Extracts adjacency structure (indptr, indices)
Computes degree per node
Builds matrix:
(n_nodes × max_degree)

 **Padding Strategy:**
Nodes with fewer neighbors are padded with -1

** Key Benefit:**
Enables fully vectorized random walks
Eliminates Python-level loops during traversal
Significantly improves performance

 **5. Edge Removal** (Cross-Validation Support)
def remove_edges(self, pairs_idx):

 **Purpose:**

Used for leakage-free cross-validation by removing selected edges.

**Steps:**
Converts CSR → COO format
Builds edge set for removal (both directions)
Applies boolean mask filtering
Reconstructs a new CSR matrix
Returns a new SparseGraph instance

**Why it matters:**

Ensures:

No test edges influence training graph
Proper cross-validation integrity
Fully leakage-free evaluation setup

 **6. Graph Construction Helper**

def build_graph(edges_df, nodes_df):
 **Workflow:**
Extract node IDs from nodes_df
Initialize SparseGraph
Map node IDs → integer indices
Filter invalid edges
Build adjacency matrix

 **Output:**

A fully constructed SparseGraph ready for ML pipeline use

 **7. Node Metadata Extraction**
def get_node_kinds(nodes_df):
 **Purpose:**

Creates a mapping:

node_id → node_type (kind)
 **Example Types:**
Gene → Protein
Compound → Drug
Disease → Condition

**Importance:**

Essential for:

Heterogeneous graph learning
Type-aware embeddings
Biological interpretation of results

**🧠 Final Insight**

The SparseGraph design provides:

✔ Massive memory savings vs NetworkX

✔ Fast CSR-based graph operations

✔ Vectorized random walk compatibility

✔ Safe cross-validation via edge removal

✔ Structured support for heterogeneous biology graphs

In [ ]:
# 4. VECTORIZED JACCARD SIMILARITY (GPU via CuPy when available)
#    Computed strictly on G_train inside each fold -> no leakage.
# ─────────────────────────────────────────────
def jaccard_similarity_dense(adj_csr, idx_subset):
    """
    Dense Jaccard similarity for a (typically small, e.g. Compound or
    Disease) subset of nodes, computed via one matrix multiplication:
        intersection = A_sub @ A_sub.T
        union        = deg_i + deg_j - intersection
        jaccard      = intersection / union
    Runs on GPU (CuPy) if available, else NumPy/SciPy.
    """
    sub = adj_csr[idx_subset, :].astype(np.float32)  # (k, n)
    deg = np.asarray(sub.sum(axis=1)).ravel()         # (k,)

    if GPU_AVAILABLE:
        sub_gpu = csp.csr_matrix(sub)
        inter_gpu = sub_gpu.dot(sub_gpu.T)            # (k, k) sparse on GPU
        inter = cp.asnumpy(inter_gpu.toarray())
    else:
        inter = sub.dot(sub.T).toarray()

    union = deg[:, None] + deg[None, :] - inter
    with np.errstate(divide='ignore', invalid='ignore'):
        jac = np.where(union > 0, inter / union, 0.0)
    np.fill_diagonal(jac, 1.0)
    return jac.astype(np.float32)


def build_similarity_matrices(G, node_kinds, kind_a='Compound', kind_b='Disease'):
    """Returns dense jaccard matrices + the local node-id lists, all index-aligned."""
    drugs = [n for n in G.node_ids if node_kinds.get(n) == kind_a]
    diseases = [n for n in G.node_ids if node_kinds.get(n) == kind_b]
    drug_idx = np.array([G.id2idx[n] for n in drugs])
    disease_idx = np.array([G.id2idx[n] for n in diseases])

    drug_jac = jaccard_similarity_dense(G.adj, drug_idx) if len(drug_idx) else np.zeros((0, 0))
    disease_jac = jaccard_similarity_dense(G.adj, disease_idx) if len(disease_idx) else np.zeros((0, 0))

    # Pre-normalize rows to probability distributions (rows that sum to 0 stay 0).
    def row_normalize(mat):
        row_sum = mat.sum(axis=1, keepdims=True)
        out = np.zeros_like(mat)
        nz = row_sum[:, 0] > 0
        out[nz] = mat[nz] / row_sum[nz]
        return out

    drug_probs = row_normalize(drug_jac)
    disease_probs = row_normalize(disease_jac)

    return {
        "drugs": drugs, "diseases": diseases,
        "drug_idx": drug_idx, "disease_idx": disease_idx,
        "drug_probs": drug_probs, "disease_probs": disease_probs,
    }



**📄 Vectorized Jaccard Similarity (GPU-Accelerated)**

This section implements an efficient, leakage-safe, and hardware-adaptive method for computing pairwise Jaccard similarity matrices for biological node subsets (e.g., compounds and diseases). The design prioritizes vectorization, numerical stability, and optional GPU acceleration.

 **1. Dense Jaccard Similarity Function**
def jaccard_similarity_dense(adj_csr, idx_subset):

**Purpose**

Computes pairwise Jaccard similarity between nodes using a matrix-based formulation, avoiding slow pairwise Python loops.

**📐 Mathematical Formulation**

The similarity is computed as:

**Intersection:**

A⋅A
T

**Union:**

deg(i)+deg(j)−intersection

**Final Jaccard Score:**

J(i,j)=
union
intersection
	​

 **2. Subgraph Extraction**
sub = adj_csr[idx_subset, :].astype(np.float32)

 **What happens:**

Extracts only selected nodes (e.g., drugs or diseases)
Keeps full adjacency context for similarity computation
Converts matrix to float32 for performance

 **3. Degree Computation**

deg = np.asarray(sub.sum(axis=1)).ravel()

 **Purpose:**
Computes number of connections per node
Used to calculate the union term in Jaccard similarity

 **4. GPU Acceleration (Optional)**

**if GPU_AVAILABLE:**

 If GPU is available (CuPy):
Converts sparse matrix to CuPy CSR format
Performs matrix multiplication on GPU:
inter_gpu = sub_gpu.dot(sub_gpu.T)
Transfers result back to CPU (NumPy)

**If GPU is NOT available:**

Falls back to NumPy/SciPy CPU computation

**✔ Ensures hardware-independent execution**

 **5. Union Computation**

union = deg[:, None] + deg[None, :] - inter

 **What this does:**

Expands degree vector into a full matrix
Computes union for every node pair efficiently

 **6. Numerical Stability Handling**

with np.errstate(divide='ignore', invalid='ignore'):

 **Purpose:**

Prevents warnings from division by zero
Ensures stable execution
jac = np.where(union > 0, inter / union, 0.0)

 **Logic:**

If union > 0 → compute Jaccard
Else → assign 0

 **7. Self-Similarity Fix**

np.fill_diagonal(jac, 1.0)
Ensures every node has similarity = 1 with itself
Maintains mathematical correctness

 **8. Similarity Matrix Builder**

def build_similarity_matrices(G, node_kinds, kind_a='Compound', kind_b='Disease'):
 **Purpose**

Constructs structured similarity matrices for specific biological entity types.

 **Node Type Filtering**

drugs = [n for n in G.node_ids if node_kinds.get(n) == kind_a]
diseases = [n for n in G.node_ids if node_kinds.get(n) == kind_b]
Separates nodes by biological type:
Compounds (Drugs)
Diseases

 **Index Mapping**

drug_idx = np.array([G.id2idx[n] for n in drugs])
Converts node IDs → integer indices
Required for matrix-based computation

 **Jaccard Matrix Construction**

drug_jac = jaccard_similarity_dense(G.adj, drug_idx)
disease_jac = jaccard_similarity_dense(G.adj, disease_idx)
Computes similarity separately for:
Drugs
Diseases

 **Important:**

Computed inside each training fold only
Prevents data leakage

 **9. Row Normalization**

def row_normalize(mat):

 **Purpose**

Converts similarity scores into probability distributions

📌 Formula:
P(i,j)=
∑
j
	​

J(i,j)
J(i,j)


 **Steps:**

Computes row sums
Normalizes each row
Safely handles zero row

**10. Output Structure**

return {
    "drugs": drugs,
    "diseases": diseases,
    "drug_idx": drug_idx,
    "disease_idx": disease_idx,
    "drug_probs": drug_probs,
    "disease_probs": disease_probs,
}

 **Returned Components:**

Node lists (drugs, diseases)
Index mappings (for matrix operations)
Normalized similarity matrices (probability form)

**🧠 Final Insight**

This implementation achieves:

✔ Fully vectorized Jaccard computation (no loops)

✔ GPU acceleration via CuPy (optional)

✔ Numerical stability and safe execution

✔ Leakage-free design (fold-based computation)

✔ Probability-ready similarity outputs

In [15]:
# 5. FULLY VECTORIZED RANDOM WALKS
#    All walks advance one step at a time, together, via NumPy ops.
# ─────────────────────────────────────────────
def vectorized_sample_from_probs(probs_matrix, rows, local_idx_lookup_list, rng):
    """
    For each entry in `rows` (a global-graph node index that is also a row
    in probs_matrix after lookup), sample a column index according to that
    row's probability distribution, via inverse-CDF (vectorized).
    Returns the sampled LOCAL column indices (into local_idx_lookup_list).
    Rows with all-zero probability return -1 (caller should fall back to graph walk).
    """
    cdf = np.cumsum(probs_matrix[rows], axis=1)          # (m, k)
    totals = cdf[:, -1]
    u = rng.random(len(rows)) * np.maximum(totals, 1e-12)
    sampled_cols = (cdf < u[:, None]).sum(axis=1)         # searchsorted, vectorized
    sampled_cols = np.clip(sampled_cols, 0, probs_matrix.shape[1] - 1)
    sampled_cols[totals <= 0] = -1
    return sampled_cols


def generate_all_walks_vectorized(G, num_walks, walk_length, teleport_prob,
                                   sim_data, node_kinds, p=1.0, q=1.0, seed=42):
    """
    Vectorized batched random walk generator with DREAMwalk teleportation.
    - All `num_walks * n_nodes` walks are advanced together, step by step.
    - Teleport candidates are drawn from the precomputed dense Compound/Disease
      Jaccard probability rows (sim_data), built on G_train only.
    - Graph steps use uniform sampling among neighbors (exact for p=q=1, the
      CONFIG default). For p!=1 or q!=1, a slower per-walk 2nd-order-biased
      fallback (`_biased_step_fallback`) is used automatically.
    """
    rng = np.random.default_rng(seed)
    n = G.n
    neighbor_idx = G.neighbor_idx
    neighbor_cnt = G.neighbor_cnt
    max_deg = neighbor_idx.shape[1]

    # Index lookups: global graph idx -> row in drug/disease prob matrices (-1 if N/A)
    drug_row_of = np.full(n, -1, dtype=np.int64)
    for local_i, gidx in enumerate(sim_data["drug_idx"]):
        drug_row_of[gidx] = local_i
    disease_row_of = np.full(n, -1, dtype=np.int64)
    for local_i, gidx in enumerate(sim_data["disease_idx"]):
        disease_row_of[gidx] = local_i

    drug_local_to_global = sim_data["drug_idx"]
    disease_local_to_global = sim_data["disease_idx"]

    base_order = np.arange(n)
    total_walks = num_walks * n
    walks = np.empty((total_walks, walk_length), dtype=np.int64)

    biased = (p != 1.0) or (q != 1.0)

    w = 0
    for _ in range(num_walks):
        rng.shuffle(base_order)
        walks[w:w + n, 0] = base_order
        w += n

    prev = np.full(total_walks, -1, dtype=np.int64)
    cur = walks[:, 0].copy()

    for step in range(1, walk_length):
        nxt = np.empty(total_walks, dtype=np.int64)

        is_drug = drug_row_of[cur] >= 0
        is_disease = disease_row_of[cur] >= 0
        teleport_roll = rng.random(total_walks) < teleport_prob
        teleport_drug = is_drug & teleport_roll
        teleport_disease = is_disease & teleport_roll & (~teleport_drug)
        graph_step = ~(teleport_drug | teleport_disease)

        if teleport_drug.any():
            idx = np.where(teleport_drug)[0]
            rows = drug_row_of[cur[idx]]
            local_cols = vectorized_sample_from_probs(sim_data["drug_probs"], rows, drug_local_to_global, rng)
            fallback = local_cols < 0
            sampled_global = np.where(fallback, -1, drug_local_to_global[np.clip(local_cols, 0, None)])
            nxt[idx[~fallback]] = sampled_global[~fallback]
            graph_step[idx[fallback]] = True  # fall back to graph step

        if teleport_disease.any():
            idx = np.where(teleport_disease)[0]
            rows = disease_row_of[cur[idx]]
            local_cols = vectorized_sample_from_probs(sim_data["disease_probs"], rows, disease_local_to_global, rng)
            fallback = local_cols < 0
            sampled_global = np.where(fallback, -1, disease_local_to_global[np.clip(local_cols, 0, None)])
            nxt[idx[~fallback]] = sampled_global[~fallback]
            graph_step[idx[fallback]] = True

        if graph_step.any():
            idx = np.where(graph_step)[0]
            cur_g = cur[idx]
            deg = neighbor_cnt[cur_g]
            dead = deg == 0
            if not biased:
                rnd = (rng.random(len(idx)) * np.maximum(deg, 1)).astype(np.int64)
                rnd = np.clip(rnd, 0, np.maximum(deg - 1, 0))
                chosen = neighbor_idx[cur_g, rnd]
            else:
                chosen = _biased_step_fallback(G, cur_g, prev[idx], rng, p, q)
            chosen[dead] = cur_g[dead]  # stuck node: stay in place
            nxt[idx] = chosen

        prev = cur
        cur = nxt
        walks[:, step] = cur

    return walks, G  # caller maps ids -> strings for Word2Vec


def _biased_step_fallback(G, cur_g, prev_g, rng, p, q):
    """Per-row (still vectorized over the batch, just not over neighbor lists)
    node2vec-style 2nd-order biased sampling, used only when p!=1 or q!=1."""
    out = np.empty(len(cur_g), dtype=np.int64)
    for i in range(len(cur_g)):
        c, pv = cur_g[i], prev_g[i]
        deg = G.neighbor_cnt[c]
        if deg == 0:
            out[i] = c
            continue
        neigh = G.neighbor_idx[c, :deg]
        if pv < 0:
            out[i] = neigh[rng.integers(0, deg)]
            continue
        weights = np.where(
            neigh == pv, 1.0 / p,
            np.where(np.isin(neigh, G.neighbor_idx[pv, :G.neighbor_cnt[pv]]), 1.0, 1.0 / q),
        )
        probs = weights / weights.sum()
        out[i] = rng.choice(neigh, p=probs)
    return out


def walks_to_str(walks, node_ids):
    id_arr = np.array(node_ids, dtype=object)
    return [list(map(str, id_arr[row])) for row in walks]


def train_embeddings(walks, config):
    model = Word2Vec(
        sentences=walks, vector_size=config['embed_dim'], window=config['window'],
        min_count=config['min_count'], sg=config['sg'], workers=config['workers'],
        epochs=config['epochs'], seed=config['random_seed'],
    )
    return model


def get_embedding(model, node_id):
    key = str(node_id)
    if key in model.wv:
        return model.wv[key]
    return np.zeros(model.vector_size)


def get_embeddings_batch(model, node_ids):
    dim = model.vector_size
    out = np.zeros((len(node_ids), dim), dtype=np.float32)
    for i, nid in enumerate(node_ids):
        key = str(nid)
        if key in model.wv:
            out[i] = model.wv[key]
    return out






**📄 Fully Vectorized Random Walks & Embeddings**

This section implements a high-performance random walk engine for heterogeneous graphs, combined with Word2Vec-based embedding learning. The design focuses on full vectorization, GPU-friendly probability sampling, and eliminating Python-level loops for scalability.

 **1. Vectorized Probability Sampling**

def vectorized_sample_from_probs(...)

 **Purpose**

Efficiently samples next nodes from probability distributions using a vectorized inverse CDF approach, avoiding loops entirely.


**Step 1: CDF Construction**

cdf = np.cumsum(probs_matrix[rows], axis=1)
Converts probabilities → cumulative distribution
Enables direct sampling without iteration

 **Step 2: Random Sampling**

u = rng.random(len(rows)) * np.maximum(totals, 1e-12)
Generates uniform random values per walk
Includes numerical stability safeguard (1e-12)

 **Step 3: Inverse CDF Selection**

sampled_cols = (cdf < u[:, None]).sum(axis=1)
Finds first index where CDF exceeds threshold
Fully vectorized replacement for searchsorted

**Output**

Returns selected column indices
Uses

 **1 for invalid or zero-probability rows**

 **2. Fully Vectorized Random Walk Generator**

def generate_all_walks_vectorized(...)

 **Goal**

Generate all random walks simultaneously, instead of per-node execution.

 **Core Idea**

Instead of:

node loop → step loop → sample neighbor

We do:

ALL walks updated together at each step

 **3. Initialization Phase**

 **RNG Setup**

rng = np.random.default_rng(seed)
Ensures reproducibility across runs

 **Graph Structures**

neighbor_idx, neighbor_cnt
Precomputed padded adjacency lists
Enable O(1) neighbor access

**Node Type Mapping**

drug_row_of, disease_row_of
Maps:
Global node → similarity matrix row
Used for teleportation decisions

 **4. Walk Initialization**

walks = np.empty((total_walks, walk_length))
Each node starts multiple independent walks
First step is randomly shuffled node assignment

**5. Main Random Walk Loop**

for step in range(1, walk_length):

At each step, all walks are updated in parallel.

 **Step 1: Node Type Detection**

is_drug, is_disease
Dynamically identifies node category
Enables type-aware transitions

**Step 2: Teleportation Decision**

teleport_roll = rng.random(...) < teleport_prob
Behavior:
Some walks:
Jump via similarity matrix (semantic move)
Others:
Continue graph traversal

 Adds global semantic exploration

**Step 3: Similarity-Based Teleportation**

vectorized_sample_from_probs(...)
Uses precomputed Jaccard similarity matrices
Applied separately for:
Drugs
Diseases

 **Ensures biologically meaningful jumps**

 **Step 4: Graph Transition**

neighbor_idx[cur_g, rnd]
Case A: Unbiased Walk (p = q = 1)
Uniform neighbor sampling
Fully vectorized
Case B: Biased Walk (Node2Vec-style)
_biased_step_fallback(...)

Uses:

p (return parameter) → controls backtracking
q (in-out parameter) → controls exploration depth

 **Step 5: Stuck Node Handling**

chosen[dead] = cur_g[dead]
If node has no neighbors:
Stay in place
Prevents invalid transitions

 **Step 6: Walk Update**

walks[:, step] = cur
Stores all walk positions simultaneously
Maintains full vectorized execution

 **6. Biased Random Walk Fallback**

def _biased_step_fallback(...)

 **Purpose**

Implements Node2Vec second-order transition rules

**Transition Rules**

For each candidate neighbor:

Return edge → weight = 1/p
Stay local → weight = 1
Explore outward → weight = 1/q

**Produces structured exploration behavior**

 **7. Conversion to Word2Vec Format**
def walks_to_str(...)
Converts numeric walks → string sequences
Required for Word2Vec training

**Example:**

[12, 45, 78] → ["12", "45", "78"]

 **8. Embedding Training (Word2Vec)**

def train_embeddings(...)

 **Purpose**

Learns node embeddings from walk sequences

 **Model Configuration**

Skip-gram (sg=1) → predicts context nodes
window → context size
vector_size → embedding dimension
epochs → training iterations

 **Output**

Dense vector representations of nodes
Capture:
Graph structure
Semantic relationships

 **9. Embedding Extraction**

Single Node
get_embedding(...)
Returns embedding vector
Defaults to zero vector if missing
Batch Extraction
get_embeddings_batch(...)
Efficient vectorized lookup
Used in downstream ML models (e.g., XGBoost)

**🧠 Final Insight**

This module achieves:

✔ Fully vectorized random walks (no loops)

✔ GPU-compatible probability sampling

✔ Hybrid graph + semantic teleportation

✔ Node2Vec-style biased exploration

✔ Scalable Word2Vec embedding learning

✔ High-performance batch embedding extraction

In [ ]:
# ─────────────────────────────────────────────
# 6. LEAK-FREE CROSS VALIDATION
#    Similarity matrices are rebuilt from G_train EVERY fold.
# ─────────────────────────────────────────────
def train_and_evaluate_clean(G, treats_df, node_kinds, config):
    drugs_all = [n for n in G.node_ids if node_kinds.get(n) == 'Compound']
    diseases_all = [n for n in G.node_ids if node_kinds.get(n) == 'Disease']

    pos_pairs = list(zip(treats_df['source'], treats_df['target']))
    pos_labels = [1] * len(pos_pairs)

    rng = np.random.default_rng(config['random_seed'])
    pos_set = set(pos_pairs)
    neg_pairs = []
    while len(neg_pairs) < len(pos_pairs):
        c = rng.choice(drugs_all)
        d = rng.choice(diseases_all)
        if (c, d) not in pos_set:
            neg_pairs.append((c, d))
    neg_labels = [0] * len(neg_pairs)

    all_pairs = np.array(pos_pairs + neg_pairs, dtype=object)
    all_labels = np.array(pos_labels + neg_labels)

    cv = StratifiedKFold(n_splits=config['n_folds'], shuffle=True, random_state=config['random_seed'])
    scaler = StandardScaler()

    aurocs, auprcs, accs = [], [], []
    results = []

    print(f"\n[eval] Starting {config['n_folds']}-fold CV WITHOUT Data Leakage "
          f"(similarity matrices rebuilt from G_train every fold)...")

    for fold, (train_idx, test_idx) in enumerate(cv.split(all_pairs, all_labels), 1):
        print(f"\n--- Processing Fold {fold}/{config['n_folds']} ---")
        pairs_train, pairs_test = all_pairs[train_idx], all_pairs[test_idx]
        y_train, y_test = all_labels[train_idx], all_labels[test_idx]

        # 1) Remove TEST-POSITIVE CtD edges from the graph -> G_train #update done
        import copy
        G_train = copy.deepcopy(G) if hasattr(G, 'copy') else G

        remove_idx_pairs = []
        for (u, v), y in zip(pairs_test, y_test):
            if y == 1 and u in G_train.id2idx and v in G_train.id2idx:
                remove_idx_pairs.append((G_train.id2idx[u], G_train.id2idx[v]))

        if remove_idx_pairs:
           res = G_train.remove_edges(remove_idx_pairs)
           if res is not None:
               G_train = res
        sim_data = build_similarity_matrices(G_train, node_kinds)
        # 3) Generate walks + embeddings on the clean graph only
        print(f"  [Fold {fold}] Generating vectorized random walks...")
        walks_idx, _ = generate_all_walks_vectorized(
            G_train, config['num_walks'], config['walk_length'], config['teleport_prob'],
            sim_data, node_kinds, config['p'], config['q'], seed=config['random_seed'] + fold,
        )
        walks_str = walks_to_str(walks_idx, G_train.node_ids)

        print(f"  [Fold {fold}] Training Word2Vec Embeddings...")
        embed_model_train = train_embeddings(walks_str, config)

        X_train = np.hstack([
            get_embeddings_batch(embed_model_train, pairs_train[:, 0]),  #error
            get_embeddings_batch(embed_model_train, pairs_train[:, 1]),
        ])
        X_test = np.hstack([
            get_embeddings_batch(embed_model_train, pairs_test[:, 0]),
            get_embeddings_batch(embed_model_train, pairs_test[:, 1]),
        ])

        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        clf = xgb.XGBClassifier(**config['xgb_params'])
        clf.fit(X_train_scaled, y_train, eval_set=[(X_test_scaled, y_test)], verbose=False)

        probs = clf.predict_proba(X_test_scaled)[:, 1]
        preds = clf.predict(X_test_scaled)

        auroc = roc_auc_score(y_test, probs)
        auprc = average_precision_score(y_test, probs)
        acc = accuracy_score(y_test, preds)

        aurocs.append(auroc); auprcs.append(auprc); accs.append(acc)
        results.append({'fold': fold, 'auroc': auroc, 'auprc': auprc, 'acc': acc})
        print(f"  --> Fold {fold} -> AUROC: {auroc:.4f} | AUPR: {auprc:.4f} | Acc: {acc:.4f}")

    print("\n" + "═" * 40)
    print(f"Mean Results (No Leakage): AUROC={np.mean(aurocs):.4f} | AUPR={np.mean(auprcs):.4f}")
    print("═" * 40)

    # Final production model: trained on the FULL graph (no test edges to hide
    # at this stage, since this model is for future, unlabeled, prediction).
    print("\nTraining final model on all data for future prediction...")
    sim_data_full = build_similarity_matrices(G, node_kinds)
    walks_idx_full, _ = generate_all_walks_vectorized(
        G, config['num_walks'], config['walk_length'], config['teleport_prob'],
        sim_data_full, node_kinds, config['p'], config['q'], seed=config['random_seed'],
    )
    walks_str_full = walks_to_str(walks_idx_full, G.node_ids)
    embed_model_all = train_embeddings(walks_str_full, config)

    X_all = np.hstack([
        get_embeddings_batch(embed_model_all, all_pairs[:, 0]),
        get_embeddings_batch(embed_model_all, all_pairs[:, 1]),
    ])
    X_all_scaled = scaler.fit_transform(X_all)

    final_clf = xgb.XGBClassifier(**config['xgb_params'])
    final_clf.fit(X_all_scaled, all_labels, verbose=False)

    return results, final_clf, scaler, embed_model_all, drugs_all, diseases_all


**📄 Leak-Free Cross-Validation & Full Training Pipeline**

This section implements a strictly leak-free evaluation framework for the DREAMwalk model, followed by training a final production-ready classifier on the full dataset. The pipeline ensures reliable performance estimation and prevents any information leakage between training and testing.

 **1. Dataset Preparation**

 **Positive Samples (Ground Truth)**
pos_pairs = list(zip(treats_df['source'], treats_df['target']))
Extracts known Compound–Disease (CtD) interactions
Represents positive class (label = 1)
 **Negative Sampling**
Randomly generates (Compound, Disease) pairs
Ensures sampled pairs are not in the positive set

 **Result:**

Positive = real biological interactions
Negative = assumed non-interactions

 **Final Dataset Construction**

all_pairs = np.array(pos_pairs + neg_pairs)
all_labels = np.array(pos_labels + neg_labels)
Combines positive + negative samples
Produces a binary supervised learning dataset

 **2. Stratified Cross-Validation Setup**

cv = StratifiedKFold(n_splits=config['n_folds'])

**Purpose:**

Maintains class balance across folds
Ensures fair and stable evaluation

**3. Leak-Free Fold Processing**

Each fold follows a strict isolation pipeline.

 **Step 1 — Remove Test Edges (Critical Leakage Prevention)**
G_train = G.remove_edges(remove_idx_pairs)
Removes test CtD edges from graph
Prevents model from indirectly seeing test information

 **This is the core anti-leakage mechanism**

 **Step 2 — Fold-Specific Similarity Computation**

sim_data = build_similarity_matrices(G_train, node_kinds)
Jaccard similarity computed only on training graph
No reuse of global precomputed matrices

**✔ Ensures full training isolation**

 **Step 3 — Random Walk Generation**
generate_all_walks_vectorized(...)
Generates fully vectorized walks
Uses:
Graph structure (local transitions)
Similarity matrices (teleportation)

 **Output:**

Node sequence corpus for embedding training

 **Step 4 — Word2Vec Embedding Training**

train_embeddings(...)
Learns embeddings from walk corpus
Captures:
Structural relationships
Semantic similarity

 S**tep 5 — Feature Construction**

np.hstack([emb_u, emb_v])
Concatenates embeddings of (u, v)
Produces edge-level feature vectors

 **Step 6 — Feature Scaling**

scaler.fit_transform(X_train)
Normalizes feature space

 **Step 7 — XGBoost Training**

clf = xgb.XGBClassifier(...)
Trains classifier on:
Embedding-based features
Binary CtD labels

 **Step 8 — Evaluation Metrics**

ROC-AUC → ranking quality
AUPRC → performance on imbalanced data
Accuracy → classification correctness

 **Step 9 — Store Fold Results**

results.append(...)
Stores metrics per fold
Enables statistical performance analysis

 **4. Final Model Training (Production Phase)**

After cross-validation, a final model is trained on the entire dataset.

 **Step 1 — Full Graph Similarity**

sim_data_full = build_similarity_matrices(G, node_kinds)
Uses complete graph (no train/test split)

 **Step 2 — Full Random Walk Corpus**

generate_all_walks_vectorized(...)
Generates maximum coverage walk corpus

 **Step 3 — Final Embedding Model**

train_embeddings(...)
Produces final node embeddings for production use

**Step 4 — Full Feature Matrix**
X_all = np.hstack(...)
Builds complete Compound–Disease feature space

 **Step 5 — Final Classifier Training**

final_clf.fit(...)
Trained on all labeled data
Used for real-world prediction

 **5. Final Outputs**

return results, final_clf, scaler, embed_model_all, drugs_all, diseases_all

 **Returned Components:**

Cross-validation results (metrics per fold)
Final trained XGBoost model
Feature scaler
Final Word2Vec embedding model
Drug and disease node lists

**Key Contributions of the Pipeline**

 **1. Strict No-Leakage Design**

Test edges removed per fold
Similarity recomputed each time
Fully isolated training pipeline

 **2. End-to-End Learning Flow**

Graph → Random Walks → Embeddings → Features → Classifier

 **3. Production-Ready Model**

Final model trained on full dataset
Ready for inference on unseen pairs

 **4. Robust Evaluation**

Stratified K-Fold CV
Multiple metrics (AUROC, AUPRC, Accuracy)
Reliable performance estimation

**🧠 Final Insight**

This pipeline is a complete graph machine learning system that combines:

Structural graph learning
Semantic embedding extraction
Supervised classification
Strict leakage-free evaluation
Production deployment readiness

In [17]:
# 7. VECTORIZED FULL PREDICTION (Compound x Disease matrix, no nested loop)
# ─────────────────────────────────────────────
def predict_all_pairs(drugs, diseases, final_clf, scaler, model, treats_df, top_k=20, batch_size=200_000):
    known = set(zip(treats_df['source'], treats_df['target']))
    drug_list = [d for d in drugs if str(d) in model.wv]
    disease_list = [d for d in diseases if str(d) in model.wv]
    print(f"\n[predict] Scoring {len(drug_list):,} x {len(disease_list):,} pairs (vectorized)...")

    drug_emb = get_embeddings_batch(model, drug_list)        # (D, dim)
    disease_emb = get_embeddings_batch(model, disease_list)  # (M, dim)

    D, M = len(drug_list), len(disease_list)
    drug_idx_grid, disease_idx_grid = np.meshgrid(np.arange(D), np.arange(M), indexing='ij')
    drug_idx_flat = drug_idx_grid.ravel()
    disease_idx_flat = disease_idx_grid.ravel()

    all_scores = np.empty(D * M, dtype=np.float32)
    total = D * M
    for start in tqdm(range(0, total, batch_size), desc="Scoring batches"):
        end = min(start + batch_size, total)
        d_idx = drug_idx_flat[start:end]
        s_idx = disease_idx_flat[start:end]
        X = np.hstack([drug_emb[d_idx], disease_emb[s_idx]])
        X_scaled = scaler.transform(X)
        all_scores[start:end] = final_clf.predict_proba(X_scaled)[:, 1]

    drug_arr = np.array(drug_list, dtype=object)[drug_idx_flat]
    disease_arr = np.array(disease_list, dtype=object)[disease_idx_flat]
    pred_df = pd.DataFrame({'compound': drug_arr, 'disease': disease_arr, 'score': all_scores})

    is_known = pred_df.apply(lambda r: (r['compound'], r['disease']) in known, axis=1)
    pred_df = pred_df[~is_known].sort_values('score', ascending=False)

    print(f"\n[predict] Top {top_k} novel drug repurposing candidates:")
    print(pred_df.head(top_k).to_string(index=False))
    return pred_df





**📄 Fully Vectorized Compound × Disease Prediction**
This section implements a fully vectorized prediction pipeline for scoring all possible Compound–Disease pairs in the DREAMwalk framework. The design avoids nested loops entirely, making large-scale biomedical inference faster, more scalable, and suitable for drug repurposing tasks.

**1) Input Filtering and Known Interaction Removal**

The pipeline first builds a set of all known Compound–Disease (CtD) interactions from the training data. These known links are later removed from the prediction results to ensure that the model only proposes novel candidate associations.

In addition, the drug and disease node lists are filtered so that only nodes with available embeddings in the trained Word2Vec model are retained. This guarantees that every predicted pair has a valid vector representation.

**2) Embedding Extraction**

Next, the model retrieves the learned embedding vectors for all valid compounds and diseases.

Drug embeddings are stored in a matrix of shape (D × embedding_dim)
Disease embeddings are stored in a matrix of shape (M × embedding_dim)

Each row corresponds to a biological entity represented in the learned latent space.

**3) Full Pairwise Candidate Generation**

To score every possible compound–disease combination, the pipeline constructs a complete Compound × Disease grid using vectorized indexing.

This replaces slow nested loops with a much more efficient approach, generating all candidate pairs in matrix form and flattening them into index arrays for downstream batch processing.

**4) Batch-Based Prediction**

Because the full pairwise space can be extremely large, prediction is performed in batches to remain memory efficient.

For each batch:

A subset of drug and disease indices is selected

Their embeddings are concatenated to form edge-level feature vectors

**[drug embedding∣disease embedding]**

The features are normalized using the same scaler fitted during training
The final XGBoost classifier predicts the probability of interaction for each pair

This design allows scalable inference without exceeding memory limits.

**5) Reconstruction of the Prediction Table**

After scoring all batches, the predicted probabilities are mapped back to their original biological entities. A final prediction table is then created with the following structure:

**Compound	Disease	Score**

This step restores the semantic meaning of the predictions by converting matrix indices back into node identifiers.

**6) Removal of Known Interactions**

All known drug–disease interactions already present in the original dataset are removed from the prediction table.

This is a critical step because it ensures that the model outputs only new, previously unseen candidate associations, rather than rediscovering known training examples.

**7) Ranking Candidate Associations**

The remaining predictions are sorted by their interaction probability scores in descending order.

As a result, the highest-confidence compound–disease pairs appear at the top of the table, making them easy to prioritize for downstream biological analysis or validation.

**8) Top-K Prediction Extraction**

Finally, the ranked table can be truncated to the Top-K predictions, returning the most promising candidate drug–disease associations.

These top-ranked predictions are particularly useful for:

Drug repurposing discovery
Biomedical hypothesis generation
Experimental validation planning

 **Final Output**

The function returns a DataFrame containing:

Predicted Compound–Disease pairs
Their corresponding interaction probability scores
Only novel candidates, after excluding known CtD interactions

 **Key Contributions of This Module
 Fully Vectorized Inference**

All possible Compound–Disease pairs are scored without nested loops, greatly reducing runtime.

 **Embedding-Based Prediction Space**

Predictions are made in a learned latent space, allowing the model to capture hidden biological relationships.

 **Memory-Safe Batch Processing**

Large pairwise prediction spaces are handled efficiently using batch-based inference.

 **Drug Repurposing Ready Output**

The final ranked list of novel drug–disease associations can be used directly in biomedical discovery workflows.

In [ ]:
#Explainable AI
import shap
import matplotlib.pyplot as plt

def explain_predictions(final_clf, scaler, model, pred_df, top_k=5):
    """
    Computes and visualizes SHAP values for the top predicted drug-disease pairs.
    Provides both Global and Local Explainable AI insights.
    """
    print("\n" + "="*50)
    print(" 🧠 Launching Explainable AI (XAI) Pipeline via SHAP")
    print("="*50)
    
    # 1. Take the top_k pairs that the model scored highest
    top_pairs = pred_df.head(top_k)
    
    # Reconstruct their features just like we did in prediction
    drug_emb = get_embeddings_batch(model, top_pairs['compound'].tolist())
    disease_emb = get_embeddings_batch(model, top_pairs['disease'].tolist())
    X_top = np.hstack([drug_emb, disease_emb])
    X_top_scaled = scaler.transform(X_top)
    
    # Create feature names for better tracking (e.g., Drug_Dim_0, Disease_Dim_0)
    dim = model.vector_size
    feature_names = [f"Drug_Dim_{i}" for i in range(dim)] + [f"Disease_Dim_{i}" for i in range(dim)]
    
    # 2. Initialize SHAP TreeExplainer for XGBoost
    explainer = shap.TreeExplainer(final_clf)
    
    print(f"[XAI] Calculating SHAP values for the top {top_k} predictions...")
    shap_values = explainer(X_top_scaled)
    
    # Assign feature names to shap values object for clean plots
    shap_values.feature_names = feature_names

    # ---------------------------------------------------------
    # A. GLOBAL EXPLANATION: Summary Plot
    # Shows which embedding dimensions drive the model's decisions overall
    # ---------------------------------------------------------
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_top_scaled, feature_names=feature_names, show=False)
    plt.title(f"SHAP Summary for Top Candidates (Top {top_k} Candidates)", fontsize=14)
    plt.tight_layout()
    summary_plot_path = os.path.join(CONFIG['cache_dir'], "shap_global_summary.png")
    plt.savefig(summary_plot_path)
    plt.close()
    print(f"✅ Global XAI Plot saved to: {summary_plot_path}")

    # ---------------------------------------------------------
    # B. LOCAL EXPLANATION: Waterfall Plot for the #1 Top Candidate
    # Explains EXACTLY why the top prediction got its high score
    # ---------------------------------------------------------
    print(f"[XAI] Generating local explanation for the top #1 candidate...")
    best_candidate = top_pairs.iloc[0]
    
    plt.figure(figsize=(12, 8))
    # shap_values[0] corresponds to the first row (the highest scoring pair)
    shap.plots.waterfall(shap_values[0], show=False)
    plt.title(f"Why {best_candidate['compound']} was linked to {best_candidate['disease']} (Score: {best_candidate['score']:.4f})", fontsize=12)
    plt.tight_layout()
    local_plot_path = os.path.join(CONFIG['cache_dir'], "shap_local_top1_explanation.png")
    plt.savefig(local_plot_path)
    plt.close()
    print(f"✅ Local XAI Plot saved to: {local_plot_path}")
    print("="*50 + "\n")

In [ ]:
# ─────────────────────────────────────────────
# 8.(Metapath-Based Explanation)
# ─────────────────────────────────────────────

# دول أكواد العلاقات (metaedges) في الـ Hetionet اللي بتربط الدوا بالجينات، والمرض بالجينات
COMPOUND_GENE_EDGES = ['CbG', 'CuG', 'CdG']   # الدوا بيرتبط/بيرفع/بيقلل نشاط الجين
DISEASE_GENE_EDGES  = ['DaG', 'DuG', 'DdG']   # المرض مرتبط/بيرفع/بيقلل نشاط الجين
GENE_PATHWAY_EDGE    = 'GpPW'                  # الجين مشترك في مسار حيوي معين


def build_gene_lookup_tables(edges_df):
    """
    بنبني هنا 3 قواميس (dictionaries) هنستخدمهم بعد كده:
    1. كل دوا -> مجموعة الجينات اللي بيأثر فيها
    2. كل مرض -> مجموعة الجينات المرتبطة بيه
    3. كل جين -> مجموعة المسارات الحيوية اللي هو جزء منها
    """
    compound_to_genes = {}
    disease_to_genes = {}
    gene_to_pathways = {}

    cg_edges = edges_df[edges_df['metaedge'].isin(COMPOUND_GENE_EDGES)]
    for src, tgt in zip(cg_edges['source'], cg_edges['target']):
        compound_to_genes.setdefault(src, set()).add(tgt)

    dg_edges = edges_df[edges_df['metaedge'].isin(DISEASE_GENE_EDGES)]
    for src, tgt in zip(dg_edges['source'], dg_edges['target']):
        disease_to_genes.setdefault(src, set()).add(tgt)

    gp_edges = edges_df[edges_df['metaedge'] == GENE_PATHWAY_EDGE]
    for src, tgt in zip(gp_edges['source'], gp_edges['target']):
        gene_to_pathways.setdefault(src, set()).add(tgt)

    return compound_to_genes, disease_to_genes, gene_to_pathways


def get_shared_evidence(compound_id, disease_id, compound_to_genes, disease_to_genes, gene_to_pathways):
    """بترجع الجينات المشتركة والمسارات الحيوية المشتركة بين دوا ومرض معينين"""
    drug_genes = compound_to_genes.get(compound_id, set())
    disease_genes = disease_to_genes.get(disease_id, set())

    shared_genes = drug_genes & disease_genes

    drug_pathways = set()
    for g in drug_genes:
        drug_pathways |= gene_to_pathways.get(g, set())

    disease_pathways = set()
    for g in disease_genes:
        disease_pathways |= gene_to_pathways.get(g, set())

    shared_pathways = drug_pathways & disease_pathways
    return shared_genes, shared_pathways


def id_to_name_map(nodes_df):
    """تحويل أي معرف (id) لاسمه الحقيقي، مثلاً DB00945 يتحول لـ Aspirin"""
    return dict(zip(nodes_df['id'], nodes_df['name']))


def explain_via_metapaths(top_pairs, edges_df, nodes_df, top_n_genes=5, top_n_pathways=5):
    """
    لكل ترشيح من الترشيحات العالية، بنطلع الجينات المشتركة والمسارات الحيوية المشتركة
    بين الدوا والمرض، عشان نديله تفسير بيولوجي مفهوم مش بس أرقام إحصائية
    """
    print("\n" + "="*50)
    print(" 🧬 التفسير البيولوجي عن طريق المسارات المشتركة")
    print("="*50)

    compound_to_genes, disease_to_genes, gene_to_pathways = build_gene_lookup_tables(edges_df)
    name_of = id_to_name_map(nodes_df)

    records = []
    for _, row in top_pairs.iterrows():
        c_id, d_id, score = row['compound'], row['disease'], row['score']
        shared_genes, shared_pathways = get_shared_evidence(
            c_id, d_id, compound_to_genes, disease_to_genes, gene_to_pathways
        )

        gene_names = [name_of.get(g, g) for g in list(shared_genes)[:top_n_genes]]
        pathway_names = [name_of.get(p, p) for p in list(shared_pathways)[:top_n_pathways]]

        records.append({
            'compound': name_of.get(c_id, c_id),
            'disease': name_of.get(d_id, d_id),
            'score': score,
            'n_shared_genes': len(shared_genes),
            'shared_genes_sample': gene_names,
            'n_shared_pathways': len(shared_pathways),
            'shared_pathways_sample': pathway_names,
        })

        print(f"\n💊 {name_of.get(c_id, c_id)}  ←→  🦠 {name_of.get(d_id, d_id)}  (Score: {score:.4f})")
        print(f"   عدد الجينات المشتركة: {len(shared_genes)}  | أمثلة: {gene_names}")
        print(f"   عدد المسارات الحيوية المشتركة: {len(shared_pathways)}  | أمثلة: {pathway_names}")

    evidence_df = pd.DataFrame(records)
    evidence_path = os.path.join(CONFIG['cache_dir'], "metapath_evidence.csv")
    evidence_df.to_csv(evidence_path, index=False)
    print(f"\n✅ جدول الأدلة البيولوجية اتسيف هنا: {evidence_path}")
    print("="*50 + "\n")
    return evidence_df


# ─────────────────────────────────────────────
# 9. التفسير العام الحقيقي (True Global Explanation)
# ─────────────────────────────────────────────

def explain_global(final_clf, scaler, model, all_pairs, all_labels, sample_size=200, seed=42):
    """
    التفسير العام الحقيقي: بياخد عينة عشوائية فيها حالات ناجحة (positive)
    وحالات فاشلة (negative) مع بعض، مش بس أعلى النتائج
    """
    print("\n" + "="*50)
    print(" 🌍 التفسير العام الحقيقي (عينة عشوائية، ناجح وفاشل مع بعض)")
    print("="*50)

    rng = np.random.default_rng(seed)
    n_sample = min(sample_size, len(all_pairs))
    idx = rng.choice(len(all_pairs), size=n_sample, replace=False)
    sample_pairs = all_pairs[idx]

    X = np.hstack([
        get_embeddings_batch(model, sample_pairs[:, 0]),
        get_embeddings_batch(model, sample_pairs[:, 1]),
    ])
    X_scaled = scaler.transform(X)

    dim = model.vector_size
    feature_names = [f"Drug_Dim_{i}" for i in range(dim)] + [f"Disease_Dim_{i}" for i in range(dim)]

    explainer = shap.TreeExplainer(final_clf)
    shap_values = explainer(X_scaled)
    shap_values.feature_names = feature_names

    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_scaled, feature_names=feature_names, show=False)
    plt.title(f"التفسير العام الحقيقي لأهمية الخصائص (عينة عشوائية n={n_sample})", fontsize=14)
    plt.tight_layout()
    global_plot_path = os.path.join(CONFIG['cache_dir'], "shap_true_global.png")
    plt.savefig(global_plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ رسمة التفسير العام اتسيفت هنا: {global_plot_path}")
    print("="*50 + "\n")
    return shap_values

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def save_pipeline_plots(results, pred_df, config):
    """
    Generates and saves research-grade plots:
    1. CV Performance Bar Plot
    2. Prediction Scores Distribution (Density Plot)
    """
    print("\n📊 Generating and saving pipeline evaluation plots...")
    res_df = pd.DataFrame(results)
    
    # ─── الجراف الأول: مقارنة المقاييس عبر الـ Folds ───
    plt.figure(figsize=(8, 5))
    melted_df = res_df.melt(id_vars='fold', value_vars=['auroc', 'auprc', 'acc'], 
                            var_name='Metric', value_name='Value')
    sns.barplot(data=melted_df, x='Metric', y='Value', errorbar="sd", palette="viridis")
    plt.title("Model Performance Metrics across 10-Folds", fontsize=12)
    plt.ylim(0, 1.05)
    plt.ylabel("Score")
    plot1_path = os.path.join(config['cache_dir'], "cv_metrics_performance.png")
    plt.savefig(plot1_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Performance Plot saved to: {plot1_path}")

    # ─── الجراف الثاني: توزيع درجات التنبؤ للمركبات الجديدة ───
    plt.figure(figsize=(8, 5))
    sns.kdeplot(pred_df['score'], fill=True, color="blue", bw_adjust=0.5)
    plt.axvline(x=0.5, color='red', linestyle='--', label='Default Threshold (0.5)')
    plt.title("Distribution of Scores for Novel Drug-Disease Pairs", fontsize=12)
    plt.xlabel("Predicted Repurposing Probability (Score)")
    plt.ylabel("Density")
    plt.legend()
    plot2_path = os.path.join(config['cache_dir'], "predictions_distribution.png")
    plt.savefig(plot2_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Prediction Distribution Plot saved to: {plot2_path}")

In [ ]:
# 8. MAIN PIPELINE
def main():
    print("=" * 55)
    print("  DREAMwalk Optimized Pipeline (Leak-Free + Vectorized)")
    print(f"  GPU similarity backend (CuPy) available: {GPU_AVAILABLE}")
    print("=" * 55)
    nodes_df, edges_df, treats_df = load_hetionet_local()
    node_kinds = get_node_kinds(nodes_df)
    G = build_graph(edges_df, nodes_df)

    results, final_clf, scaler, embed_model_all, drugs, diseases, all_pairs, all_labels = train_and_evaluate_clean(
        G, treats_df, node_kinds, CONFIG
    )

    results_df = pd.DataFrame(results)
    results_df.to_csv(os.path.join(CONFIG['cache_dir'], "clean_cv_results.csv"), index=False)

    pred_df = predict_all_pairs(drugs, diseases, final_clf, scaler, embed_model_all, treats_df, top_k=20)
    pred_df.to_csv(os.path.join(CONFIG['cache_dir'], "clean_predictions.csv"), index=False)

    # 1️⃣ تفسير SHAP لأعلى الترشيحات
    explain_predictions(final_clf, scaler, embed_model_all, pred_df, top_k=10)

    # 2️⃣ التفسير البيولوجي عن طريق المسارات المشتركة
    top_pairs_for_metapaths = pred_df.head(10)
    explain_via_metapaths(top_pairs_for_metapaths, edges_df, nodes_df)

    # 3️⃣ التفسير العام الحقيقي
    explain_global(final_clf, scaler, embed_model_all, all_pairs, all_labels, sample_size=200)

    # 4️⃣ رسومات الأداء والتوزيع
    save_pipeline_plots(results, pred_df, CONFIG)

    print("\n[done] Optimized, leak-free DREAMwalk complete.")

if __name__ == "__main__":
    main()

  DREAMwalk Optimized Pipeline (Leak-Free + Vectorized)
  GPU similarity backend (CuPy) available: False
Nodes: 47,031
Edges: 2,250,197
CtD Edges: 755

[eval] Starting 10-fold CV WITHOUT Data Leakage (similarity matrices rebuilt from G_train every fold)...

--- Processing Fold 1/10 ---
  [Fold 1] Building Jaccard similarity from G_train (GPU=False)...
  [Fold 1] Generating vectorized random walks...
  [Fold 1] Training Word2Vec Embeddings...
  --> Fold 1 -> AUROC: 0.9263 | AUPR: 0.9363 | Acc: 0.8874

--- Processing Fold 2/10 ---
  [Fold 2] Building Jaccard similarity from G_train (GPU=False)...
  [Fold 2] Generating vectorized random walks...
  [Fold 2] Training Word2Vec Embeddings...
  --> Fold 2 -> AUROC: 0.9135 | AUPR: 0.9136 | Acc: 0.8344

--- Processing Fold 3/10 ---
  [Fold 3] Building Jaccard similarity from G_train (GPU=False)...
  [Fold 3] Generating vectorized random walks...
  [Fold 3] Training Word2Vec Embeddings...
  --> Fold 3 -> AUROC: 0.9391 | AUPR: 0.9357 | Acc: 0.8874

Scoring batches: 100%|██████████| 2/2 [00:00<00:00,  2.58it/s]



[predict] Top 20 novel drug repurposing candidates:
         compound             disease    score
Compound::DB00620 Disease::DOID:10763 0.999565
Compound::DB00620 Disease::DOID:10283 0.999500
Compound::DB00741 Disease::DOID:10763 0.999414
Compound::DB00635 Disease::DOID:10763 0.999399
Compound::DB00860 Disease::DOID:10763 0.999395
Compound::DB00544 Disease::DOID:10763 0.999384
Compound::DB00443 Disease::DOID:10763 0.999324
Compound::DB01234 Disease::DOID:10763 0.999289
Compound::DB00290 Disease::DOID:10763 0.999277
Compound::DB00620  Disease::DOID:3393 0.999273
Compound::DB01229  Disease::DOID:2531 0.999258
Compound::DB00620  Disease::DOID:1612 0.999136
Compound::DB00541  Disease::DOID:1612 0.999133
Compound::DB00091  Disease::DOID:2531 0.999119
Compound::DB00860  Disease::DOID:3393 0.999084
Compound::DB00175 Disease::DOID:10763 0.999078
Compound::DB01229 Disease::DOID:10283 0.998999
Compound::DB00773  Disease::DOID:1612 0.998958
Compound::DB00860 Disease::DOID:10283 0.998936
Compoun


**Fully Vectorized Compound × Disease Prediction**
This section implements a highly optimized inference pipeline that evaluates all possible Compound–Disease pairs in a fully vectorized way. The design avoids nested loops entirely and is built for large-scale biomedical prediction and drug repurposing.

 **1. Input Filtering & Known Interaction Removal**

known = set(zip(treats_df['source'], treats_df['target']))

 **Purpose**

Stores all known Compound–Disease (CtD) interactions
Used to exclude existing relationships from predictions

 **Valid Node Filtering**

drug_list = [d for d in drugs if str(d) in model.wv]

 **Logic:**
Keeps only nodes present in the Word2Vec vocabulary
Ensures all prediction inputs have valid embeddings

**✔Output:**

Clean drug list
Clean disease list

**2.Embedding Extraction**

drug_emb = get_embeddings_batch(model, drug_list)

 **Purpose**
Converts node IDs → embedding vectors

**📐Shapes:**

Drugs → (D × embedding_dim)
Diseases → (M × embedding_dim)


**Each row represents a biological entity in latent semantic space**

 **3. Full Pairwise Grid Construction**

 **Purpose**
Generates all possible Compound × Disease combinations

 **Key Advantage:**

Replaces nested loops with vectorized indexing
Produces D × M candidate pairs efficiently

 **4.Batch-Based Prediction Strategy**

for start in range(0, total, batch_size):

 **Why batching?**

Prevents memory overflow
Enables scalable inference over large graphs

 **4.1Pair Selection**

d_idx = drug_idx_flat[start:end]
s_idx = disease_idx_flat[start:end]
Selects subset of pairs per batch

**4.2 Feature Construction**

X = np.hstack([drug_emb[d_idx], disease_emb[s_idx]])

 **Meaning:**

Each pair becomes:

[drug_embedding | disease_embedding]

**📐Shape:**

(batch_size × 2 × embedding_dim)

 **4.3 Feature Scaling**

X_scaled = scaler.transform(X)

 **Purpose:**
Ensures consistency with training distribution
Critical for stable XGBoost predictions

**4.4 Model Inference**

final_clf.predict_proba(X_scaled)
Outputs probability of interaction for each pair
Stored in all_scores

 **5.Reconstruction of Prediction Table**

pred_df = pd.DataFrame(...)

**Purpose**

Converts numerical predictions into interpretable format:

| compound | disease | score |

 **Index Mapping**

drug_arr = np.array(drug_list)[drug_idx_flat]
Converts indices → original biological IDs
Restores semantic meaning of predictions

 **6. Removal of Known Interactions**

pred_df = pred_df[~is_known]

 **Purpose:**

Removes existing CtD edges
Ensures only novel predictions remain

 **Prevents trivial re-discovery of training data**

 **7.Ranking Predictions**

pred_df.sort_values('score', ascending=False)

 **Behavior:**

Highest probability pairs appear first
Enables prioritization for biological validation

 **8.Top-K Candidate Extraction**

pred_df.head(top_k)

 **Purpose:**

Returns top predicted interactions for:

 Drug repurposing
 Biological hypothesis generation
 Experimental validation

 **9.Final Output**

return pred_df
 **Output Contains:**

All predicted Compound–Disease pairs
Interaction probability scores
Filtered novel candidates

 **Key Innovations**

 **1. Fully Vectorized Prediction**

No loops
O(D × M) operations handled efficiently

 **2.Embedding-Based Latent Space Prediction**

Predictions performed in learned vector space
Captures hidden biological relationships

 **3.Batch Inference Engine**

Memory-safe design
Scales to large biomedical graphs

**4.Drug Repurposing Ready Output**

Ranked novel candidates
Directly usable for discovery pipelines

**🧠 Final Insight**
This module transforms the DREAMwalk model into a real-world biomedical prediction engine by:

Generating all possible drug–disease pairs efficiently
Scoring them in embedding space
Filtering known interactions
Producing ranked novel hypotheses

In [20]:
import pandas as pd
print(pd.read_csv("dreamwalk_cache/clean_cv_results.csv"))

   fold     auroc     auprc       acc
0     1  0.926316  0.936316  0.887417
1     2  0.913509  0.913556  0.834437
2     3  0.939123  0.935705  0.887417
3     4  0.958772  0.962869  0.894040
4     5  0.954035  0.955006  0.867550
5     6  0.948070  0.951950  0.894040
6     7  0.927895  0.916011  0.854305
7     8  0.947719  0.951629  0.880795
8     9  0.957368  0.950321  0.907285
9    10  0.923684  0.904588  0.867550


In [ ]:
import pandas as pd

df = pd.read_csv("dreamwalk_cache/clean_cv_results.csv")

print("--- Full Cross-Validation Results ---")
print(df)
print("-" * 50)

print("--- Overall Average Scores ---")
 
metrics_cols = [col for col in ['AUROC', 'AUPR', 'Acc', 'Accuracy'] if col in df.columns]

if metrics_cols:
    averages = df[metrics_cols].mean()
    for metric, val in averages.items():
        print(f"Average {metric}: {val:.4f}")
else:
    print(df.mean(numeric_only=True))

--- Full Cross-Validation Results ---
   fold     auroc     auprc       acc
0     1  0.926316  0.936316  0.887417
1     2  0.913509  0.913556  0.834437
2     3  0.939123  0.935705  0.887417
3     4  0.958772  0.962869  0.894040
4     5  0.954035  0.955006  0.867550
5     6  0.948070  0.951950  0.894040
6     7  0.927895  0.916011  0.854305
7     8  0.947719  0.951629  0.880795
8     9  0.957368  0.950321  0.907285
9    10  0.923684  0.904588  0.867550
--------------------------------------------------
--- Overall Average Scores ---
fold     5.500000
auroc    0.939649
auprc    0.937795
acc      0.877483
dtype: float64


**📄 Code Explanation — Displaying Cross-Validation Results**

import pandas as pd
print(pd.read_csv("dreamwalk_cache/clean_cv_results.csv"))

This code is used to load and display the cross-validation results that were previously saved during the DREAMwalk evaluation stage.

 **1. Importing Pandas**

import pandas as pd
Imports the Pandas library, which is used for handling structured data in tabular form.
Pandas provides the DataFrame object, making it easy to read, organize, and analyze datasets such as CSV files.

**2. Reading the Results File**

pd.read_csv("dreamwalk_cache/clean_cv_results.csv")
Reads the file clean_cv_results.csv from the dreamwalk_cache directory.
This file contains the evaluation results generated during the cross-validation phase of the DREAMwalk pipeline.
pd.read_csv() loads the CSV file into a DataFrame, where the data is organized into rows and columns for easy inspection.

 **3. Printing the DataFrame**

print(...)
Displays the contents of the DataFrame directly in the console or notebook output.
This allows quick inspection of the performance metrics obtained from each cross-validation fold.

 **What the File Typically Contains**

The clean_cv_results.csv file usually stores one row per cross-validation fold, along with the corresponding evaluation metrics. Common columns include:

fold → the fold number in cross-validation
auroc → Area Under the ROC Curve
auprc → Area Under the Precision–Recall Curve
acc → classification accuracy

**Purpose of This Code**

This snippet is mainly used for reviewing model performance after training and evaluation. It provides a simple way to inspect how the DREAMwalk model performed across all folds and compare the consistency of its predictive results.